In [1]:
import os
import yaml
import copy

import numpy as np
import pandas as pd
import xarray as xr

In [2]:
import xesmf as xe

In [3]:
import matplotlib.pyplot as plt
%matplotlib inline

In [4]:
def _sedi_from_hits_falsealarms(H, F, eps=1e-6):
    """
    H, F are DataArrays in [0,1]. Apply clipping to avoid log(0) and log(1-0).
    """
    Hc = H.clip(eps, 1 - eps)
    Fc = F.clip(eps, 1 - eps)

    num = (np.log(Fc) - np.log(Hc) - np.log(1 - Fc) + np.log(1 - Hc))
    den = (np.log(Fc) + np.log(Hc) + np.log(1 - Fc) + np.log(1 - Hc))
    return num / den

def _compute_sedi_binary(fcst, obs, dim, event_quantile=2/3):
    """
    Compute SEDI for deterministic forecasts, using an observation-based threshold.

    fcst, obs: DataArrays aligned on 'dim' and spatial dims.
    event_quantile: 2/3 = upper tercile (paper); 0.8 = 80th percentile (paper's ENSO-removed case).
    """
    # threshold from observations, per-gridcell
    thr = obs.quantile(event_quantile, dim=dim, skipna=True)

    obs_event  = obs  > thr
    fcst_event = fcst > thr

    # contingency components across time dim
    hits        = (fcst_event & obs_event).sum(dim=dim)
    misses      = (~fcst_event & obs_event).sum(dim=dim)
    falsealarms = (fcst_event & ~obs_event).sum(dim=dim)
    correctneg  = (~fcst_event & ~obs_event).sum(dim=dim)

    # rates
    H = hits / (hits + misses)
    F = falsealarms / (falsealarms + correctneg)

    sedi = _sedi_from_hits_falsealarms(H, F)
    return sedi

### Pituffik: melting degree days

In [5]:
key = 'Pituffik'
base_dir = f'/glade/derecho/scratch/ksha/EPRI_data/METRICS/{key}/'

In [6]:
fn_CESM = base_dir + 'MDD.zarr'
fn_ERA5 = base_dir + 'ERA5_MDD.zarr'

ds_CESM = xr.open_zarr(fn_CESM)
ds_ERA5 = xr.open_zarr(fn_ERA5)

In [7]:
ds_CESM = ds_CESM.rename({'init_time': 'year_valid'})
ds_ERA5 = ds_ERA5.rename({'year': 'year_valid'})

ds_CESM = ds_CESM.chunk(dict(year_valid=-1))
ds_ERA5 = ds_ERA5.chunk(dict(year_valid=-1))


varnames = list(ds_ERA5.keys())

acc_list_all_leads = []
sedi_list_all_leads = []

for i_lead in range(10):
    ds_CESM_slice = ds_CESM.isel(year=i_lead)
    # convert ini_time to valid_time by adding lead times
    ds_CESM_slice['year_valid'] = ds_CESM_slice['year_valid'] + i_lead

    # align two datasets
    ds_ERA5_slice = ds_ERA5
    ds_CESM_slice, ds_ERA5_slice = xr.align(ds_CESM_slice, ds_ERA5_slice, join="inner")

    if ds_CESM_slice.dims.get("year_valid", 0) >= 10:
        acc_vars = []
        sedi_vars = []

        for v in varnames:
            fcst = ds_CESM_slice[v]
            obs  = ds_ERA5_slice[v]

            # ACC
            acc = xr.corr(fcst, obs, dim="year_valid")
            acc = acc.rename(f"{v}_ACC")
            acc_vars.append(acc)

            # SEDI (upper tercile events, as in the paper’s categorical setup)
            sedi = _compute_sedi_binary(fcst, obs, dim="year_valid", event_quantile=2/3)
            sedi = sedi.rename(f"{v}_SEDI")
            sedi_vars.append(sedi)

        ds_metrics = xr.merge(acc_vars + sedi_vars)
        
        # keep a lead coordinate so concat is clean
        ds_metrics = ds_metrics.expand_dims(lead=[i_lead])
        acc_list_all_leads.append(ds_metrics)

ds_ACC_SEDI_all = xr.concat(acc_list_all_leads, dim="lead")

In [8]:
save_name = base_dir + 'ACC_MDD.zarr'
# ds_ACC_SEDI_all.to_zarr(save_name, mode='w')
print(save_name)

/glade/work/ksha/miniconda3/envs/credit/lib/python3.11/site-packages/dask/_task_spec.py:759: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)


/glade/derecho/scratch/ksha/EPRI_data/METRICS/Pituffik/ACC_MDD.zarr


/glade/work/ksha/miniconda3/envs/credit/lib/python3.11/site-packages/dask/_task_spec.py:759: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)


### Pituffik: Freeze-Thaw days

In [9]:
fn_CESM = base_dir + 'FT.zarr'
fn_ERA5 = base_dir + 'ERA5_FT.zarr'

ds_CESM = xr.open_zarr(fn_CESM)
ds_ERA5 = xr.open_zarr(fn_ERA5)

In [10]:
ds_CESM = ds_CESM.rename({'init_time': 'year_valid'})
ds_ERA5 = ds_ERA5.rename({'year': 'year_valid'})

ds_CESM = ds_CESM.chunk(dict(year_valid=-1))
ds_ERA5 = ds_ERA5.chunk(dict(year_valid=-1))


varnames = list(ds_ERA5.keys())

acc_list_all_leads = []
sedi_list_all_leads = []

for i_lead in range(10):
    ds_CESM_slice = ds_CESM.isel(year=i_lead)
    # convert ini_time to valid_time by adding lead times
    ds_CESM_slice['year_valid'] = ds_CESM_slice['year_valid'] + i_lead

    # align two datasets
    ds_ERA5_slice = ds_ERA5
    ds_CESM_slice, ds_ERA5_slice = xr.align(ds_CESM_slice, ds_ERA5_slice, join="inner")

    if ds_CESM_slice.dims.get("year_valid", 0) >= 10:
        acc_vars = []
        sedi_vars = []

        for v in varnames:
            fcst = ds_CESM_slice[v]
            obs  = ds_ERA5_slice[v]

            # ACC
            acc = xr.corr(fcst, obs, dim="year_valid")
            acc = acc.rename(f"{v}_ACC")
            acc_vars.append(acc)

            # SEDI (upper tercile events, as in the paper’s categorical setup)
            sedi = _compute_sedi_binary(fcst, obs, dim="year_valid", event_quantile=2/3)
            sedi = sedi.rename(f"{v}_SEDI")
            sedi_vars.append(sedi)

        ds_metrics = xr.merge(acc_vars + sedi_vars)
        
        # keep a lead coordinate so concat is clean
        ds_metrics = ds_metrics.expand_dims(lead=[i_lead])
        acc_list_all_leads.append(ds_metrics)

ds_ACC_SEDI_all = xr.concat(acc_list_all_leads, dim="lead")

In [11]:
# plt.pcolormesh(ds_ACC_SEDI_all['FT_days_SEDI'].values[9, ...], cmap=plt.cm.jet)
# plt.colorbar()

In [12]:
save_name = base_dir + 'ACC_FT.zarr'
# ds_ACC_SEDI_all.to_zarr(save_name, mode='w')
print(save_name)

/glade/derecho/scratch/ksha/EPRI_data/METRICS/Pituffik/ACC_FT.zarr
